# Question and why it matters
Does opening-run length predict late boundary occupation after adjusting for the
area reached in that same window? Earlier area is a post-treatment variable and
may leave growth-history confounding. See [registration and results](README.md)
and [mechanism context](../../../notes/kappa-mechanisms-2026-10-15.md).


## Assumptions and definitions
Kappa = (B/S)/(P/A), where B counts steps starting on visited boundary cells and
P counts visited cells with an unvisited four-neighbour. All quantities use the
probe's octave windows. Conditioning on area is descriptive and cannot estimate
a causal direct effect. Constant composition does not distinguish leading-run
length from first-L position. CPU/GPU equivalence is checked before the sweep.


In [1]:
from pathlib import Path
import hashlib, json
SEED = 2026101608
root = Path.cwd().resolve()
while not (root / 'Cargo.toml').exists():
    if root == root.parent:
        raise RuntimeError('Repository root not found')
    root = root.parent
exp = root / 'experiments/archive/2026-10-kappa-growth-clock'
paths = [exp / 'results' / s for s in ['analysis.json', 'cohort.csv', 'windows.csv', 'registration.md', 'spot-check.txt']]
for p in paths:
    assert p.is_file(), p
    print(str(p.relative_to(root)), hashlib.sha256(p.read_bytes()).hexdigest())
report = json.loads(paths[0].read_text())
print('Seed:', SEED, 'Simulation base commit:', report['git'])
print('Run commit:', (exp / 'results/run-commit.txt').read_text().strip())
print('Date: 2026-10-16 +13:00; exact raw/source hashes are in analysis.json and run-sources.sha256')


experiments/2026-10-16-kappa-growth-clock/results/analysis.json 7a1f4217c22346d6325c3566d08178305b2ea6dad4947bc6f427e5d829fbd74e
experiments/2026-10-16-kappa-growth-clock/results/cohort.csv 9bdf18c4493f1d549fd7d510b3572cdb7541273b863948c5df428b956fe7f848
experiments/2026-10-16-kappa-growth-clock/results/windows.csv c5df342a3bea1b6ce188b0c731f23a87edadb34f556a1186ad3d2484b9c08239
experiments/2026-10-16-kappa-growth-clock/results/registration.md 8e926fab77367d0c9221df239a2bc9d9dcb560ffe36781d7cbd3f2ab7a72c557
experiments/2026-10-16-kappa-growth-clock/results/spot-check.txt 2a85d93856130e93831452b6d751f0adf9f8337c5443d911da26d0b2e5d4abe3
Seed: 2026101608 Simulation base commit: c9811f56911ff4e7fd8cfe5e76cd5d461968265b
Run commit: c9811f56911ff4e7fd8cfe5e76cd5d461968265b
Date: 2026-10-16 +13:00; exact raw/source hashes are in analysis.json and run-sources.sha256


## Data and method
Sixteen nine-letter suffixes each supply four 14-letter rules with leading runs
1..4; all rules in a family have equal composition. Four patch seeds per rule,
2^24 steps, grid 4096. Primary k=23, secondary k=21,22; early area is A18.
No stationarity or exponent filtering. Require three completed seeds per rule
and twelve complete families. OLS area controls use training-fitted spline
hinges. Prediction uses leave-one-family-out folds. Uncertainty resamples whole
families. The heavy computation lives in scripts/run.sh and scripts/analyse.py;
this notebook reads only published results and makes no new estimates.


In [2]:
print('Runs:', report['runs'], 'Escaped:', report['escaped'])
print((exp / 'results/spot-check.txt').read_text().strip())
print((exp / 'results/cpu-replay-check.txt').read_text().strip())
print(json.dumps(report['calibration'], indent=2))
print('Attrition:', report['status_counts'])


Runs: 256 Escaped: 45
Four specs byte-identical: probe/gpu_probe 24 4096
Full CPU replay: all 256 specs byte-identical to frozen GPU CSV
{
  "null_slope": 1.6004895062095795e-16,
  "planted_slope": 0.2000000000000002,
  "accounting_error": 0.0,
  "noisy_repetitions": 200,
  "noisy_null_mean": 0.00021200019296161492,
  "noisy_planted_mean": 0.1502120001929615,
  "known_null_cv_gain": 2.307788105654426e-05,
  "scope": "Exact spline coefficient and identity calibration; no exponent estimator used."
}
Attrition: {'k21': {'missing_early_window': 40, 'missing_target_window': 4, 'valid': 212}, 'k22': {'missing_early_window': 40, 'missing_target_window': 4, 'valid': 212}, 'k23': {'missing_early_window': 40, 'missing_target_window': 4, 'valid': 212}}


## Results
The early and current slopes use the same response and rules, changing only the
area-control window. P1 requires the current slope to retain half the early slope
and have a positive interval. P2 requires gain >=0.05 and a positive interval.


In [3]:
for r in report['windows']:
    print('Window', r['k'], 'families', r['complete_families'], 'retention', r['qualifying_by_lead'])
    if not r['gate_passed']:
        print(r['verdict']); continue
    print('Early slope %.4f %s; current %.4f %s' % (r['early_slope'], r['early_ci'], r['current_slope'], r['current_ci']))
    print('Held-out gain %.4f %s; P1 %s; P2 %s' % (r['gain'], r['gain_ci'], r['P1'], r['P2']))
    print('Accounting: boundary %.4f - edge fraction %.4f = kappa %.4f' % (r['boundary_log_slope'], r['edge_fraction_log_slope'], r['current_slope']))
    print('Both-area linear slope %.4f; median between-seed sd(log kappa) %.4f' % (r['both_area_linear_slope'], r['between_seed_sd_median']))


Window 21 families 7 retention {'1': 16, '2': 9, '3': 15, '4': 13}
UNINFORMATIVE: complete-family support gate failed
Window 22 families 7 retention {'1': 16, '2': 9, '3': 15, '4': 13}
UNINFORMATIVE: complete-family support gate failed
Window 23 families 7 retention {'1': 16, '2': 9, '3': 15, '4': 13}
UNINFORMATIVE: complete-family support gate failed


These tables show whether the registered association survives a different
growth-clock control. They do not separate cause from response to growth history.


**Final verdict: UNINFORMATIVE.** All three windows retain seven complete
families, below the registered twelve-family gate. P1/P2 were not fitted or
scored. Of 256 starts, 45 eventually escape; 40 miss the early window and four
more miss the targets. The 212 valid starts do not supply balanced families.
This does not support or reject the lead association.


## Caveats and controls
Synthetic exact-spline and correlated-noise controls calibrate the regression;
the coefficient decomposition is an identity. CPU/GPU outputs are compared byte
for byte on all 256 specs and release tests recorded. The cohort has only sixteen independent
suffix families and uses a shorter horizon than the previous 2^27 study. Area
and perimeter are measured on the response trajectories; neither is an
independent causal control. Gain intervals condition on fitted folds. Support
failure means uninformative, and failed thresholds remain failed.


## What this changes / next steps
Read the frozen primary verdict in results/analysis.json together with the final
README interpretation. If current-area adjustment removes the association, early
area alone was insufficient at this horizon; if it survives, test transport with
independent pre-window predictors before interventions. Longer-horizon and
fixed-area replication are needed in either case.

Open questions: how much is survivor selection, which field history matters,
and can first-L position be separated from opening-run length?
